# Metadata harmonisation

Turns `Data/tags.tsv` + `Data/sample_metadata.tsv` + `Data/projects.csv` into one
tidy row per sample with trustworthy columns. The logic lives in `src/harmonize.py`;
this notebook runs it once end to end, writes `data/interim/samples_harmonized.parquet`
and `reports/harmonization_coverage.md`, and checks `disease_label` against the
hand-curated counts in `config/within_study_review.csv`.

This notebook consumes the label catalogue produced by `00b_explore.ipynb` rather than
rediscovering it. `disease_label` here is scoped to 62 curated (project, field) pairs
in `config/within_study_review.csv`: the 53 CONFIRMED within-study case/control
contrasts (the `within_project` cohort) plus 9 HEALTHY_ONLY studies added in this pass
to seed the `healthy_baseline` cohort (see Step 6). The broader `labeled_all` cohort
(116 projects) needs the remaining ~63 candidate fields triaged the same way — noted
in the closing section as the natural next task, not silently approximated.

In [1]:
import sys
import csv
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import harmonize as harm

REPORTS = ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
(ROOT / "data" / "interim").mkdir(parents=True, exist_ok=True)

with open(ROOT / "config" / "null_values.yaml") as f:
    null_cfg = yaml.safe_load(f)
with open(ROOT / "config" / "health_keywords.yaml") as f:
    keywords_cfg = yaml.safe_load(f)
with open(ROOT / "config" / "tag_map.yaml") as f:
    tag_map = yaml.safe_load(f)
with open(ROOT / "config" / "condition_map.yaml") as f:
    condition_cfg = yaml.safe_load(f)
with open(ROOT / "config" / "kit_map.yaml") as f:
    kit_cfg = yaml.safe_load(f)
review_df = pd.read_csv(ROOT / "config" / "within_study_review.csv")

print(f"tag_map: {len(tag_map)} concepts, "
      f"{sum(len(v) for v in tag_map.values())} source tags")
print(f"condition_map: {len(condition_cfg['rules'])} category rules")
print(f"kit_map: {len(kit_cfg['rules'])} family rules")
print(f"review_df: {len(review_df)} curated rows, "
      f"{(review_df['bucket'] == 'CONFIRMED').sum()} CONFIRMED")

tag_map: 7 concepts, 14 source tags
condition_map: 33 category rules
kit_map: 10 family rules
review_df: 72 curated rows, 53 CONFIRMED


## Load the raw metadata

Same three files and the same `QUOTE_NONE` reasoning as `00b_explore.ipynb` — `tags.tsv`
contains literal double-quote characters as data, which corrupts pandas' default
CSV-style quoting far past where they occur.

In [2]:
tags_df = pd.read_csv(ROOT / "Data" / "tags.tsv", sep="\t", dtype=str,
                       quoting=csv.QUOTE_NONE)
sample_metadata_df = pd.read_csv(ROOT / "Data" / "sample_metadata.tsv", sep="\t", dtype=str,
                                  quoting=csv.QUOTE_NONE)
projects_df = pd.read_csv(ROOT / "Data" / "projects.csv", encoding="utf-8-sig", dtype=str)

print("tags_df:", tags_df.shape)
print("sample_metadata_df:", sample_metadata_df.shape)
print("projects_df:", projects_df.shape)
assert len(sample_metadata_df) == 168_464

tags_df: (3489745, 5)
sample_metadata_df: (168464, 11)
projects_df: (482, 10)


## Step 1 — Pivot the concept tags, long → wide

`tag_map.yaml` lists 7 canonical concepts (age, age_unit, sex, bmi, host_species,
collection_date, subject_id) and their duplicate-named source tags (`age`/`host_age`,
`sex`/`host_sex`, ...). Pivoting is restricted to just these tags — pivoting all 2,608
would be unusably sparse. `geo` and `body_site` are deliberately not mapped: the
sample_metadata spine already carries `region`/`iso`/`geo_loc_name`, and the non-stool
filter (in `02_cohorts.ipynb`) uses `projects.csv.sample_type`, not a tags.tsv field.

In [3]:
wide = harm.pivot_concepts(tags_df, tag_map, null_cfg)
wide = wide.merge(sample_metadata_df[["srr", "project"]], on="srr", how="left")

print(f"{len(wide):,} samples pivoted")
for concept in tag_map:
    n = wide[concept].notna().sum()
    print(f"  {concept:18s} {n:>7,} non-null  (from {wide[concept + '_source_tag'].dropna().unique().tolist()})")
assert len(wide) == 168_464

168,464 samples pivoted
  age                 40,264 non-null  (from ['age', 'host_age'])
  age_unit             5,620 non-null  (from ['age_unit', 'host_age_units', 'age units', 'age unit'])
  sex                 39,498 non-null  (from ['sex', 'host_sex'])
  bmi                 14,143 non-null  (from ['host_body_mass_index', 'body_mass_index'])
  host_species       122,623 non-null  (from ['host'])
  collection_date    103,833 non-null  (from ['collection_date'])
  subject_id          45,653 non-null  (from ['host_subject_id', 'subject_id'])


## Step 2 — Repair known corruptions

`sex` has leaked ages (`"47"` × 2,794, `"48"` × 1,501) and a string-concatenation bug
(`"not providednot provided"`). `age_unit` has leaked numbers (`78` × 1,663). Both are
nulled rather than guessed.

In [4]:
sex_raw = wide["sex"]
wide["sex"], n_sex_leaked = harm.repair_sex(wide["sex"])
wide["sex_raw"] = sex_raw
print(f"sex: {n_sex_leaked:,} leaked-age values nulled; "
      f"{wide['sex'].notna().sum():,} clean male/female values remain")

wide["age_unit"], n_unit_leaked = harm.repair_age_unit(wide["age_unit"])
print(f"age_unit: {n_unit_leaked:,} leaked-number values nulled")

wide["bmi"] = harm.clean_bmi(wide["bmi"])
print(f"bmi: {wide['bmi'].notna().sum():,} numeric values")

sex: 4,295 leaked-age values nulled; 35,063 clean male/female values remain
age_unit: 2,641 leaked-number values nulled
bmi: 13,607 numeric values


## Step 3 — Parse age → `age_years` + `age_confidence`

Handles an explicit unit tag, an embedded unit (`"6 months"`), a range
(`"17-29 yo"` → midpoint), an open bound (`">=100"` → 100), and a bare numeric with no
unit anywhere — inferred per project from `projects.csv.subjects` (infants imply
months), never silently assumed to be years. Free text that matches none of these
(`"child 2-year visit"`) is left null rather than guessed.

In [5]:
age_years, age_confidence, n_out_of_range = harm.parse_ages(wide, projects_df)
wide["age_years"] = age_years
wide["age_confidence"] = age_confidence

print(f"age_years: {age_years.notna().sum():,} parsed, {n_out_of_range} out-of-range nulled")
print(age_confidence.value_counts(dropna=False).rename("count").to_string())
assert age_years.dropna().between(0, 120).all(), "age_years acceptance: 0 <= age_years <= 120"
assert age_years.notna().equals(age_confidence.notna()), \
    "every non-null age_years must carry an age_confidence"
print("PASS — age bounds and confidence-pairing hold")

age_years: 39,045 parsed, 243 out-of-range nulled
NaN         129419
inferred     30464
exact         6310
range         2070
bound          201
PASS — age bounds and confidence-pairing hold


## Step 4 — Normalise `host_species`

`host` is genuinely free text (204 distinct values, including plant names and
per-subject codes like `"human a2"`), so this is prefix/substring matching, not an
exact lookup. Values naming neither a human nor a known non-human marker are left
`<NA>` for review rather than guessed either way.

In [6]:
wide["host_species_human"] = harm.classify_host_species(wide["host_species"])
vc = wide["host_species_human"].value_counts(dropna=False)
print(vc.rename("count").to_string())

unresolved_host = wide.loc[wide["host_species_human"].isna() & wide["host_species"].notna(), "host_species"]
print(f"\n{len(unresolved_host):,} samples have a host value naming neither — top examples:")
print(unresolved_host.value_counts().head(8).to_string())

host_species_human
True     121956
<NA>      46005
False       503

164 samples have a host value naming neither — top examples:
host_species
glucorticoid-induced obesity patient and healthy individuals    54
male, 4 months, persistent diarrhea, vitamin a deficiency        6
male, 6 months, persistent diarrhea, vitamin a deficiency        4
male, 3 months, persistent diarrhea, vitamin a deficiency        4
female, 5 months, persistent diarrhea, vitamin a deficiency      3
female, 3 months, persistent diarrhea, vitamin a normal          3
female, 4 months, persistent diarrhea, vitamin a deficiency      3
male, 8 months, persistent diarrhea, vitamin a normal            3


## Step 5 — Parse `collection_date` → `collection_year`

Applies the null vocabulary first, then `pd.to_datetime`, then a fallback that takes
the leading four-digit year out of MIxS date-RANGE values (`"2011-01-01/2014-12-31"`)
that `pd.to_datetime` can't parse as a single date.

In [7]:
wide["collection_year"] = harm.parse_collection_year(wide["collection_date"], "collection_date", null_cfg)

n_null = wide["collection_year"].isna().sum()
print(f"collection_year: {wide['collection_year'].notna().sum():,} parsed, {n_null:,} null "
      f"(of {len(wide):,} samples)")
print(f"year range: {int(wide['collection_year'].min())}-{int(wide['collection_year'].max())}")
print("\nIMPLEMENTATION.md's reference figure (45,402 unparseable) turns out to conflate "
      "genuinely-missing values (\"na\", \"missing\", \"not applicable\", ...) with actual "
      "parse failures -- once the null vocabulary is applied first, only a handful of true "
      "garbage values (\"-\", \"0000-00-00\") remain unparseable; the rest of the null count "
      "here is samples with no collection_date tag at all, or a null-vocabulary value.")

collection_year: 103,822 parsed, 64,642 null (of 168,464 samples)
year range: 1919-2021

IMPLEMENTATION.md's reference figure (45,402 unparseable) turns out to conflate genuinely-missing values ("na", "missing", "not applicable", ...) with actual parse failures -- once the null vocabulary is applied first, only a handful of true garbage values ("-", "0000-00-00") remain unparseable; the rest of the null count here is samples with no collection_date tag at all, or a null-vocabulary value.


## Step 6 — Derive `disease_label`

Two curated buckets in `config/within_study_review.csv` feed this. **CONFIRMED** (53
projects, within-study case/control contrasts) uses `resolve_value_labels` — exact
subset-sum against the hand-verified controls/cases counts, not keyword matching:
most confirmed fields code the contrast as numbers (`"0"`/`"1"`), abbreviations
(`"hc"`, `"cd"`, `"uc"`) or study-specific terms (`nonrecurrent`/`reinfection`) with
no generic control/case vocabulary in them at all. **HEALTHY_ONLY** (9 projects, added
to seed the `healthy_baseline` cohort) uses `resolve_healthy_only_labels` — an
all-or-nothing check that every value in the field is a control token, since there's
no case side to reconstruct a partition against. See both functions' docstrings in
`src/harmonize.py` for why keyword matching alone was tried first and abandoned.

The 9 HEALTHY_ONLY projects were found by scanning every project whose
`projects.csv.condition` is exactly `"healthy"` (121 candidates, after excluding the 3
already in CONFIRMED that turned out to have hidden cases — PRJEB11419, PRJEB5729,
PRJNA554232) for a per-sample field where every non-null value, after null handling,
is a recognised control value. 112 of the 121 had no qualifying field (no per-sample
health confirmation at all, or a field with at least one non-control value) and are
excluded, not guessed into the cohort.

In [8]:
condition_norm = harm.normalize_condition(projects_df, condition_cfg)
disease_df, unresolved = harm.derive_disease_label(
    tags_df, null_cfg, keywords_cfg, review_df, condition_cfg, condition_norm,
)

print(f"disease_label resolved for {disease_df['project'].nunique()} projects, "
      f"{len(disease_df):,} samples")
print(disease_df["disease_label"].value_counts().rename("count").to_string())
print(f"unresolved (project, field, bucket) triples: {unresolved}")
assert not unresolved, "a curated field's current values no longer match its reviewed bucket"

# Reconcile exactly against config/within_study_review.csv, project by project --
# resolve_value_labels / resolve_healthy_only_labels only accept a mapping when
# it matches the reviewed bucket exactly, so this should never fail; asserted
# anyway as a regression check.
per_project = disease_df.groupby("project")["disease_label"].value_counts().unstack(fill_value=0)
curated = review_df[review_df["bucket"].isin(["CONFIRMED", "HEALTHY_ONLY"])]
reviewed = curated.set_index("project")[["controls", "cases"]]
check = per_project.join(reviewed)
mismatched = check[(check.get("healthy", 0) != check["controls"]) | (check.get("case", 0) != check["cases"])]
assert mismatched.empty, f"disease_label counts drifted from within_study_review.csv:\n{mismatched}"
print(f"PASS — all {len(check)} curated projects' case/control counts match "
      f"within_study_review.csv exactly")

n_confirmed = (review_df["bucket"] == "CONFIRMED").sum()
n_healthy_only = (review_df["bucket"] == "HEALTHY_ONLY").sum()
within_project_n = disease_df[disease_df["project"].isin(curated[curated["bucket"] == "CONFIRMED"]["project"])].shape[0]
healthy_only_n = disease_df[disease_df["project"].isin(curated[curated["bucket"] == "HEALTHY_ONLY"]["project"])].shape[0]
print(f"\nwithin_project (Way A, CONFIRMED): {within_project_n:,} samples across {n_confirmed} projects "
      f"(reference: 18,541 strict binary contrast / 20,819 full field population, "
      f"per notebooks/00b_explore.ipynb)")
print(f"healthy_only addition (for Phase 2's healthy_baseline): {healthy_only_n:,} samples across "
      f"{n_healthy_only} projects")

disease_label resolved for 62 projects, 19,732 samples
disease_label
healthy    12433
case        7299
unresolved (project, field, bucket) triples: []
PASS — all 62 curated projects' case/control counts match within_study_review.csv exactly

within_project (Way A, CONFIRMED): 18,541 samples across 53 projects (reference: 18,541 strict binary contrast / 20,819 full field population, per notebooks/00b_explore.ipynb)
healthy_only addition (for Phase 2's healthy_baseline): 1,191 samples across 9 projects


## Step 7 — American Gut self-report battery

23 fields (minus the excluded `fungal_overgrowth`) sharing a four-level diagnostic
vocabulary, present almost exclusively in PRJEB11419 / PRJEB5729. Emits
`sr_<field>_evidence` (ordinal 0-3) and `sr_<field>_status` (binary; only evidence==3
counts as a case — a self-diagnosis is not treated as one). These are separate from
the primary `disease_label` above, which for these two projects is resolved from the
`diabetes` / `ibd` fields specifically (Step 6).

In [9]:
self_report_df = harm.derive_self_report_columns(tags_df, keywords_cfg)
status_cols = [c for c in self_report_df.columns if c.endswith("_status")]
coverage = self_report_df[status_cols].notna().sum().sort_values(ascending=False)
print(f"{len(status_cols)} self-report fields, {coverage.sum():,} total non-null status values")
print(coverage.head(10).rename("n_samples").to_string())

26 self-report fields, 86,048 total non-null status values
sr_diabetes_status              5497
sr_ibd_status                   5374
sr_asd_status                   4626
sr_thyroid_status               4592
sr_autoimmune_status            4568
sr_skin_condition_status        4480
sr_clinical_condition_status    4465
sr_add_adhd_status              4449
sr_acid_reflux_status           4383
sr_migraine_status              4370


## Step 8 — Normalise `projects.csv`

`condition` (204 distinct free-text strings) → a controlled category via
`condition_map.yaml`; `kit` (283 distinct strings) → a manufacturer family via
`kit_map.yaml`. Both project-level, broadcast onto every sample in that project.

In [10]:
kit_norm = harm.normalize_kit(projects_df, kit_cfg)

print(f"condition_category: {condition_norm['condition_category'].nunique()} categories "
      f"from {projects_df['condition'].nunique()} raw strings")
print(condition_norm["condition_category"].value_counts().head(8).rename("n_projects").to_string())
print()
print(f"kit_family: {kit_norm['kit_family'].nunique()} families from "
      f"{projects_df['kit'].nunique()} raw strings")
print(kit_norm["kit_family"].value_counts().rename("n_projects").to_string())

condition_category: 32 categories from 204 raw strings
condition_category
other           205
healthy_only    142
obesity          14
HIV              12
CRC              10
preterm          10
cancer_other      9
T2D               8

kit_family: 9 families from 283 raw strings
kit_family
Qiagen                  236
other_or_unspecified    159
manual_extraction        32
MP_Biomedicals           26
Roche                     9
Zymo                      7
Stratec                   7
Macherey_Nagel            4
Promega                   2


## Step 9 — Join everything onto the 168,464-row spine

Left joins throughout — a sample missing a piece (no disease label, unparseable age,
...) keeps its row with nulls in those columns rather than being dropped.

In [11]:
wide_final = wide.drop(columns=["project"])
harmonized = harm.build_harmonized(
    sample_metadata_df, wide_final, disease_df, self_report_df,
    condition_norm, kit_norm, projects_df,
)

out_path = ROOT / "data" / "interim" / "samples_harmonized.parquet"
harmonized.to_parquet(out_path, index=False)
print(f"wrote {out_path.relative_to(ROOT)}: {harmonized.shape}")

assert len(harmonized) == 168_464
assert not harmonized["srr"].duplicated().any()
assert harmonized["project"].notna().all()
print("PASS — 168,464 rows, no duplicate srr, no null project")

wrote data\interim\samples_harmonized.parquet: (168464, 91)
PASS — 168,464 rows, no duplicate srr, no null project


## Step 10 — Coverage report

In [12]:
CONCEPTS = [
    "age_years", "age_confidence", "sex", "bmi", "host_species_human",
    "collection_year", "subject_id", "disease_label", "disease_category",
    "condition_category", "kit_family", "amplicon",
]
report_md = harm.coverage_report(harmonized, CONCEPTS)
(REPORTS / "harmonization_coverage.md").write_text(
    "# Phase 1 harmonisation coverage\n\n"
    f"{len(harmonized):,} samples.\n\n" + report_md + "\n",
    encoding="utf-8",
)
print(report_md)

| column | non-null | % |
|---|---|---|
| `age_years` | 39,045 | 23.2% |
| `age_confidence` | 39,045 | 23.2% |
| `sex` | 35,063 | 20.8% |
| `bmi` | 13,607 | 8.1% |
| `host_species_human` | 122,459 | 72.7% |
| `collection_year` | 103,822 | 61.6% |
| `subject_id` | 45,653 | 27.1% |
| `disease_label` | 19,732 | 11.7% |
| `disease_category` | 7,299 | 4.3% |
| `condition_category` | 168,464 | 100.0% |
| `kit_family` | 168,464 | 100.0% |
| `amplicon` | 168,464 | 100.0% |


## Summary

In [13]:
for f in sorted((ROOT / "data" / "interim").glob("*")) + sorted(REPORTS.glob("*")):
    print(f"{str(f.relative_to(ROOT)):40s} {f.stat().st_size / 1e6:8.2f} MB")

data\interim\sample_depth.parquet            1.86 MB
data\interim\samples_harmonized.parquet      4.10 MB
data\interim\taxa_full.parquet              78.10 MB
data\interim\taxa_nonzero.parquet           38.48 MB
data\interim\taxa_prev01.npz                74.76 MB
data\interim\taxon_table.parquet             0.23 MB
reports\cohort_flow.md                       0.00 MB
reports\figures                              0.00 MB
reports\harmonization_coverage.md            0.00 MB
reports\project_summary.csv                  0.03 MB


## Results and notes

### What this notebook produced

`data/interim/samples_harmonized.parquet` — one row per sample (168,464, matching the
spine exactly), with cleaned `age_years` / `sex` / `bmi` / `host_species_human` /
`collection_year`, a `disease_label` (`healthy` / `case`) with `disease_category`
covering 62 curated projects, the 23-field American Gut self-report ladder, and
project-level `condition_category` / `kit_family` / `amplicon`.
`reports/harmonization_coverage.md` records non-null coverage per column, and the
config maps this notebook touched (`tag_map.yaml`, `condition_map.yaml`,
`kit_map.yaml`, plus 9 new rows in `within_study_review.csv`) are hand-checked.

### `disease_label` covers `within_project` and seeds `healthy_baseline`

The 53 CONFIRMED pairs (within-study case/control contrasts) reconstruct exactly —
same projects, same control/case counts, verified by assertion above — so
`disease_label` restricted to them **is** the `within_project` cohort directly. 9 more
projects were added this pass as **HEALTHY_ONLY**: every project whose
`projects.csv.condition` is exactly `"healthy"` was scanned for a per-sample field
that is 100% a control value after null handling (e.g. `PRJEB5714`'s `disease` field
is `"healthy"` for all 50 samples; `PRJNA388263`'s `gastrointest_disord` is `"none"`
for 419 of 443). Of 121 candidate projects, 112 had no field that cleared this bar —
either no per-sample health tag at all, or a field with at least one value that
wasn't a recognised control token, in which case the whole project is excluded rather
than partially trusted. This is deliberately conservative and yields far fewer
samples (1,191 across 9 projects) than the ~54,571 / 125 figure an earlier estimate
computed from the study-level `condition` field alone — the expected shrinkage once
`healthy_baseline` is rebuilt from per-sample labels instead.

Neither addition reaches the broader `labeled_all` set (116 projects, ~31,239
samples): the label catalogue (`health_field_catalogue.csv`) has ~63 further
candidate fields still bucketed `NEEDS_REVIEW`, and auto-assigning labels to them
without the same value-level check would repeat the "wrong tag" / "confident false
positive" failure modes the manual review documents. Extending
`within_study_review.csv` to cover them, the same way both the 53 and the 9 were
curated, is the direct next step.

### The value-to-label mapping problem, and why it isn't keyword matching

An early version of `disease_label` used a keyword classifier (control/case
vocabulary from `health_keywords.yaml`). It silently mis-scored several confirmed
fields, because most of them don't use recognisable words at all: `PRJNA554232`'s
`env_local_scale` is coded `"0"` / `"1"`, `PRJNA641414`'s `Case_Status` is the same,
and `PRJNA307992`'s C. diff `patient_group` splits `nonrecurrent` (control) from
`reinfection` (case) — a direction no generic vocabulary rule could infer from the
text. `resolve_value_labels` replaces that with exact subset-sum against the reviewed
`controls` / `cases` totals: it finds the partition of a field's value distribution
that reproduces the hand-verified counts, rather than guessing from the words. All 53
CONFIRMED fields resolved with zero mismatches once one more null-vocabulary gap was
found and fixed: PRJEB5729's `ibd` field uses `"no"` for its 803 controls, which the
original `null_values.yaml` exemption list didn't cover (only the MIxS `*_disord`
family and `host_disease` were exempted) — added as a named exemption with the
reasoning inline. The 9 HEALTHY_ONLY fields use a stricter, simpler check
(`resolve_healthy_only_labels`): since there's no case side to reconstruct a
partition against, every value in the field must be a control token, or the whole
field is rejected.

### Two other corrections found while building this

`collection_date`'s null-vocabulary list was missing `"not available"` — a large
chunk of the earlier "unparseable" count was actually this un-recognised null string
plus a timezone-mixing bug in the naive `pd.to_datetime` call (fixed by passing
`utc=True`). With a leading-year fallback for MIxS date ranges added too, true parse
failures drop to single digits; see Step 5 for the reconciliation.

`host_species` cannot be fully resolved from the `host` tag alone: beyond the known
human / non-human variants, a meaningful share of values are demographic free text
with no species word at all (e.g. `"male, 4 months, persistent diarrhea, vitamin a
deficiency"`). These are left `<NA>` rather than assumed human — the non-human
exclusion filter downstream should treat `<NA>` as "needs review", not as a pass.

### For the next notebook

`within_project` is `harmonized[harmonized["project"].isin(confirmed_projects)]` (the
53 CONFIRMED projects). `healthy_baseline` is
`harmonized[(harmonized["disease_label"] == "healthy") &
harmonized["project"].isin(healthy_only_projects)]` (the 9 HEALTHY_ONLY projects) —
not `projects.csv.condition`, which misdescribes its own contents in both directions.
Both project sets are recoverable from `config/within_study_review.csv`'s `bucket`
column; no further label engineering needed, only the depth / host / sample_type QC
filters.